In [5]:
from docplex.mp.model import Model
import pandas as pd
import random
import math
import time
import operator 
from operator import getitem 

# ==============================================================================
# ⚠️ NOTE ON INPUT DATA:
# This code assumes 'edgecbd.csv' and 'usercbd.csv' are present in the
# execution environment. Since these files are not provided, the code will
# use dummy data for 'edgeloc' and 'userloc' to ensure it runs without error.
# If you have the actual CSV files, remove the following dummy data lines.
# ==============================================================================

# Global parameters for environment size (Used to sample from locations)
num_edges = 125 
num_users = 816

# Fallback/Dummy data creation
try:
    # --- Data Loading (Assumed available: edgecbd.csv, usercbd.csv) ---
    df = pd.read_csv('edgecbd.csv', delimiter=',')
    edgeloc = [list(row) for row in df.values]

    dfu = pd.read_csv('usercbd.csv', delimiter=',')
    userloc = [list(row) for row in dfu.values]
    
except FileNotFoundError:
    print("Warning: 'edgecbd.csv' or 'usercbd.csv' not found. Using dummy location data.")
    random.seed(42) # Seed for reproducible dummy data
    edgeloc = [[random.uniform(40, 42), random.uniform(-75, -73)] for _ in range(num_edges)]
    userloc = [[random.uniform(40, 42), random.uniform(-75, -73)] for _ in range(num_users)]
    random.seed(None) # Reset seed
# ==============================================================================


def run_optimization(edges, users, services, budget, w_latency, w_availability, edgeloc, userloc, num_edges, num_users):
    """
    Sets up, solves, and reports the results for a single run, generating new
    user requests (demand) and edge/service parameters for this iteration.
    """
    
    # --- Data Generation (NEW FOR EACH ITERATION) ---
    
    # Edge Detailing (Coverage, Available Slots, Capacity)
    coverage = []
    availableservices = []
    maxrequests =[]
    for i in range(edges):
        coverage.append(random.randint(200,1000)) 
        availableservices.append(random.randint(2, services))
        maxrequests.append(random.randint(50,150))

    # Service Detailing (Cost, Latency Limit, Reliability)
    cost = []
    latlim = []
    service_reliability = [0.9, 0.99, 0.995, 0.9999, 0.99999]
    service_avail = random.choices(service_reliability, k=services)

    for i in range(services):
        cost.append(random.randint(40,120))
        latlim.append(random.randint(10, 500)) 
        
    # **USER DEMAND GENERATION (NEW FOR EACH ITERATION)**
    demand = [] # Demand[k] is the service ID (1-based) requested by user k
    for i in range(users):
        demand.append(random.randint(1, services))
        
    # Edge and User Location Selection (sampling from available locations)
    userArray = []
    a = random.sample(range(1, num_users + 1), users)
    for i in range(users):
        uloc = userloc[a[i]-1] if i < len(a) else userloc[0] 
        userArray.append(uloc)

    edgeArray = []
    a = random.sample(range(1, num_edges + 1), edges)
    for i in range(edges):
        eloc = edgeloc[a[i]-1] if i < len(a) else edgeloc[0] 
        edgeArray.append(eloc)

    # Calculate haversine distance (NEW FOR EACH ITERATION)
    randist = [] 
    R = 6371 # km
    for e in range(edges):
        distloc = []
        for u in range(users):
            lat1, lon1 = edgeArray[e]
            lat2, lon2 = userArray[u]
            dLat = (lat2 - lat1) * (math.pi/180)
            dLon = (lon2 - lon1) * (math.pi/180)
            a_val = math.sin(dLat/2)**2 + math.cos(lat1* (math.pi/180)) * math.cos(lat2*(math.pi/180)) * math.sin(dLon/2)**2
            c_val = 2 * math.atan2(math.sqrt(a_val), math.sqrt(1-a_val))
            d_km = R * c_val
            distloc.append(d_km * 1000) # distance in meters
        randist.append(distloc)

    # --- Create model ---
    model = Model(name="qos_aware_fault_tolerant_placement")

    # Decision variables for placement (x) and allocation (y_primary, y_backup)
    x = {(i, j): model.binary_var(name=f"x_{i}_{j}") for i in range(edges) for j in range(services)}
    y_primary = {(i, k): model.binary_var(name=f"y_primary_{i}_{k}") for i in range(edges) for k in range(users)}
    y_backup = {(i, k): model.binary_var(name=f"y_backup_{i}_{k}") for i in range(edges) for k in range(users)}

    # --- OBJECTIVE FUNCTION & CONSTRAINTS (Use the newly generated data) ---
    
    epsilon = 1e-9
    # Latency Score (depends on randist and latlim)
    latency_score = model.sum(y_primary[i, k] * (1 - ( randist[i][k] / latlim[demand[k] - 1]))
                            for i in range(edges) for k in range(users) if latlim[demand[k] - 1] > 0)

    # Availability Score (depends on service_avail and demand)
    availability_score = model.sum(y_backup[i, k] * (1 / (1 - service_avail[demand[k] - 1] + epsilon))
                                for i in range(edges) for k in range(users))

    model.maximize(w_latency * latency_score + w_availability * availability_score)

    # All constraints use the newly generated parameters (randist, latlim, coverage, cost, availableservices, maxrequests, demand)
    # 1-3. User allocation constraints (remain the same)
    for k in range(users):
        model.add_constraint(model.sum(y_primary[i, k] for i in range(edges)) <= 1)
        model.add_constraint(model.sum(y_backup[i, k] for i in range(edges)) <= 1)
        for i in range(edges):
            model.add_constraint(y_primary[i, k] + y_backup[i, k] <= 1)
        model.add_constraint(model.sum(y_backup[i, k] for i in range(edges)) <= model.sum(y_primary[i, k] for i in range(edges)))

    # 4. Service availability constraints (depends on demand)
    for i in range(edges):
        for k in range(users):
            model.add_constraint(y_primary[i, k] <= x[i, demand[k] - 1])
            model.add_constraint(y_backup[i, k] <= x[i, demand[k] - 1])

    # 5. Latency constraints (depends on randist and latlim)
    for i in range(edges):
        for k in range(users):
            model.add_constraint(randist[i][k] * y_primary[i, k] <= latlim[demand[k] - 1])
            model.add_constraint(randist[i][k] * y_backup[i, k] <= latlim[demand[k] - 1])
                                
    # 6. Coverage constraints (depends on randist and coverage)
    for i in range(edges):
        for k in range(users):
            model.add_constraint(randist[i][k] * y_primary[i, k] <= coverage[i])
            model.add_constraint(randist[i][k] * y_backup[i, k] <= coverage[i])

    # 7. Budget constraint (depends on cost)
    model.add_constraint(model.sum(cost[j] * x[i, j] for i in range(edges) for j in range(services)) <= budget, "budget")

    # 8. Server capacity constraints (depends on availableservices and maxrequests)
    for i in range(edges):
        model.add_constraint(model.sum(x[i, j] for j in range(services)) <= availableservices[i])
        model.add_constraint(model.sum(y_primary[i, k] + y_backup[i, k] for k in range(users)) <= maxrequests[i])


    # --- Solve and Report ---
    start_time = time.time()
    solution = model.solve(log_output=False) 
    end_time = time.time()
    elapsed_time = end_time - start_time
    
    totalp, totalb, totals, objective_value = 0, 0, 0, 0
    
    if solution:
        objective_value = solution.get_objective_value()
        totalp = sum(solution.get_value(y_primary[(i, k)]) for i in range(edges) for k in range(users))
        totalb = sum(solution.get_value(y_backup[(i, k)]) for i in range(edges) for k in range(users))
        totals = sum(solution.get_value(x[(i, j)]) for i in range(edges) for j in range(services))

    return totalp, totalb, totals, objective_value, elapsed_time


# ==============================================================================
# --- Iterative Run Setup ---
# ==============================================================================

# Input Configuration (from the original script's global parameters)
ITERATIONS = 10
edges = 20
users = 100
services = 4
budget = 3500
w_latency = 0.8
w_availability = 0.2

# Global lists to store results
primary_results = []
backup_results = []
service_placement_results = []
objective_results = []
time_results = []

print(f"Starting {ITERATIONS} optimization runs (each with new user requests):")
print("-" * 50)

# --- Iterative Execution ---
for run in range(ITERATIONS):
    # Ensure random seed varies between runs to generate new requests and parameters
    random.seed(time.time() + run) 
    
    totalp, totalb, totals, obj_val, elapsed_time = run_optimization(
        edges, users, services, budget, w_latency, w_availability, edgeloc, userloc, num_edges, num_users
    )
    
    if obj_val > 0: # Only record successful solutions
        primary_results.append(totalp / users * 100)
        backup_results.append(totalb / users * 100)
        service_placement_results.append(totals)
        objective_results.append(obj_val)
        time_results.append(elapsed_time)
        print(f"Run {run + 1}/{ITERATIONS}: Primary: {primary_results[-1]:.2f}%, Backup: {backup_results[-1]:.2f}%, Services: {service_placement_results[-1]:.2f}, Obj: {objective_results[-1]:.2f}, Time: {time_results[-1]:.2f}s")
    else:
        # Note: If no solution is found, it means the new random request set 
        # made the problem infeasible under the given constraints/budget.
        print(f"Run {run + 1}/{ITERATIONS}: Solution not found (Infeasible or unbounded). Time: {elapsed_time:.2f}s")

print("-" * 50)

# --- Final Output: Average Results ---
if primary_results:
    avg_primary = sum(primary_results) / len(primary_results)
    avg_backup = sum(backup_results) / len(backup_results)
    avg_services = sum(service_placement_results) / len(service_placement_results)
    avg_objective = sum(objective_results) / len(objective_results)
    avg_time = sum(time_results) / len(time_results)
    
    print(f"✅ **Average Results over {len(primary_results)} successful runs:**")
    print(f"  Average Primary Assignment Coverage: **{avg_primary:.2f}%**")
    print(f"  Average Backup Assignment Coverage: **{avg_backup:.2f}%**")
    print(f"  Average Number of Services Placed: **{avg_services:.2f}**")
    print(f"  Average Objective Value: **{avg_objective:.2f}**")
    print(f"  Average Execution Time: **{avg_time:.4f} seconds**")
else:
    print("❌ No successful solutions were found across all runs.")

Starting 10 optimization runs (each with new user requests):
--------------------------------------------------
Run 1/10: Primary: 58.00%, Backup: 33.00%, Services: 44.00, Obj: 340016.00, Time: 1.01s
Run 2/10: Primary: 54.00%, Backup: 34.00%, Services: 36.00, Obj: 28292.00, Time: 0.02s
Run 3/10: Primary: 70.00%, Backup: 44.00%, Services: 47.00, Obj: 7666.71, Time: 0.03s
Run 4/10: Primary: 66.00%, Backup: 35.00%, Services: 44.00, Obj: 591964.06, Time: 0.03s
Run 5/10: Primary: 76.00%, Backup: 63.00%, Services: 49.00, Obj: 44510.71, Time: 0.03s
Run 6/10: Primary: 64.00%, Backup: 43.00%, Services: 43.00, Obj: 64827.95, Time: 0.03s
Run 7/10: Primary: 60.00%, Backup: 47.00%, Services: 43.00, Obj: 60079.61, Time: 0.03s
Run 8/10: Primary: 56.00%, Backup: 29.00%, Services: 37.00, Obj: 154.02, Time: 0.02s
Run 9/10: Primary: 59.00%, Backup: 39.00%, Services: 47.00, Obj: 48054.89, Time: 0.02s
Run 10/10: Primary: 47.00%, Backup: 21.00%, Services: 38.00, Obj: 795.06, Time: 0.03s
--------------------